# Exercício 1: PINN mínima versus Runge–Kutta (RK45)

Equipe: José Salgado, Renan Brêttas e Heitor Morais.

Este notebook roda **todo** o exercício 1 do repositório, do início ao fim, sem intervenção: baixa o código,
mostra e roda o código do exercício (célula a célula) e mostra a tabela e as figuras que o relatório usa.

**Como usar no Colab:** menu *Ambiente de execução → Executar tudo* (o Colab pede login com uma conta Google).
Leva poucos minutos em CPU.

In [ ]:
# 1. Baixa o repositório (se ainda não estivermos dentro dele) e entra na pasta.
import os, subprocess, sys

REPO = "jssppedra-png/trabalho-pinn-runge-kutta"
PASTA = REPO.split("/")[1]

if not os.path.exists("exercicio1.py"):
    if not os.path.isdir(PASTA):
        url = f"https://github.com/{REPO}.git"
        try:  # no Colab, usa o token guardado nos Secrets, se houver (necessário se o repositório for privado)
            from google.colab import userdata
            url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/{REPO}.git"
        except Exception:
            pass
        r = subprocess.run(["git", "clone", "-q", url, PASTA], capture_output=True, text=True)
        if r.returncode != 0:  # não mostra a mensagem do git, que poderia conter o token
            raise RuntimeError("Não consegui baixar o repositório. Se ele for privado, confira o GITHUB_TOKEN nos Secrets.")
    os.chdir(PASTA)

print("Pasta de trabalho:", os.getcwd())

## 2. Versões das ferramentas

Os números do relatório foram gerados com as versões de `requirements.txt` (Python 3.14, CPU). O Colab traz versões
próprias. Os erros do RK45 se repetem; os da PINN podem mudar bastante (num teste no Colab, o do problema (ii) ficou
cerca de 6 vezes maior), porque o resultado do treino depende da versão do PyTorch e do processador. A conclusão não
muda: a PINN continua muito atrás do RK45. Os tempos dependem da máquina.
Para usar exatamente as mesmas versões, mude `INSTALAR_VERSOES_EXATAS` para `True` (demora mais).

In [ ]:
INSTALAR_VERSOES_EXATAS = False
if INSTALAR_VERSOES_EXATAS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import platform, numpy, scipy, torch, matplotlib
print("Python    ", platform.python_version())
print("numpy     ", numpy.__version__)
print("scipy     ", scipy.__version__)
print("torch     ", torch.__version__)
print("matplotlib", matplotlib.__version__)

## 3. O código do exercício

O código está escrito nas próximas células, em quatro pedaços: (3.1) os três problemas e a régua de erro,
(3.2) o Runge–Kutta, (3.3) a PINN e (3.4) a comparação. É o mesmo código dos arquivos `problemas.py`, `rk.py`,
`pinn.py` e `exercicio1.py` do repositório, e dá os mesmos números.

### 3.1 Os três problemas e a régua

Cada problema é uma "ficha": a regra de mudança `f`, o ponto de partida `y0`, o intervalo de tempo e a resposta certa.
Os dois métodos leem a mesma ficha. A régua é a mesma para os dois: o erro máximo numa grade de 1000 pontos.

In [ ]:
# Os problemas do exercício 1, num lugar só, para o RK45 e a PINN usarem exatamente os mesmos.
# (Mesmo código do arquivo problemas.py.)
#
# Cada problema tem:
#   f(t, y, xp)   : a "regra de mudança" y' = f(t, y). y pode ter mais de um número (um sistema).
#   y0            : o ponto de partida y(0), uma lista com um número para cada variável
#   intervalo     : de onde até onde no tempo
#   exata(t, xp)  : a resposta certa (quando existe fórmula), usada só para medir o erro
#
# "xp" é a caixa de ferramentas de contas: numpy (no RK45) ou torch (na PINN).
# As duas têm xp.exp, xp.cos, xp.sin, xp.stack, então a mesma fórmula serve para os dois métodos.
# Convenção: a última posição de y guarda as variáveis. y[..., 0] é a primeira, y[..., 1] a segunda.
import numpy as np
from scipy.integrate import solve_ivp


def lotka_volterra(t, y, xp):
    coelhos, raposas = y[..., 0], y[..., 1]
    d_coelhos = coelhos - coelhos * raposas    # coelhos nascem; são comidos quando encontram raposas
    d_raposas = -raposas + coelhos * raposas   # raposas morrem de fome; crescem quando comem coelhos
    return xp.stack([d_coelhos, d_raposas], -1)


PROBLEMAS = {
    # Questão (i): a "colher que esfria rápido". Resposta: y = e^(-50t)
    "rigido": {
        "descricao": "y' = -50y, y(0) = 1",
        "f": lambda t, y, xp: -50 * y,
        "y0": [1.0],
        "intervalo": (0.0, 1.0),
        "exata": lambda t, xp: xp.exp(-50 * t),
    },
    # Questão (ii): algo que oscila rápido. Resposta: y = sen(20t) / 20
    "oscilante": {
        "descricao": "y' = cos(20t), y(0) = 0",
        "f": lambda t, y, xp: xp.cos(20 * t),
        "y0": [0.0],
        "intervalo": (0.0, 1.0),
        "exata": lambda t, xp: xp.sin(20 * t) / 20,
    },
    # Questão (iii): coelhos e raposas (Lotka-Volterra), um sistema não rígido.
    # Não existe fórmula exata: a "resposta certa" vem de um Runge-Kutta de altíssima precisão.
    "coelhos_raposas": {
        "descricao": "Lotka-Volterra: c' = c - c r, r' = -r + c r, c(0) = 2, r(0) = 1",
        "f": lotka_volterra,
        "y0": [2.0, 1.0],
        "intervalo": (0.0, 10.0),
        "exata": None,
    },
}

# Mesma régua para os dois métodos: erro máximo (em todas as variáveis) numa grade de 1000 pontos.
PONTOS_DA_REGUA = 1000


def grade_da_regua(nome):
    a, b = PROBLEMAS[nome]["intervalo"]
    return np.linspace(a, b, PONTOS_DA_REGUA).reshape(-1, 1)   # uma coluna de tempos


def resposta_certa(nome, t):
    """Resposta certa nos tempos t (coluna). Devolve uma tabela: uma linha por tempo, uma coluna por variável."""
    p = PROBLEMAS[nome]
    if p["exata"] is not None:
        return p["exata"](t, np)
    # Sem fórmula: referência com DOP853 (Runge-Kutta de ordem 8) e tolerância 1e-13,
    # mil vezes mais exigente que a tolerância dos métodos comparados.
    ref = solve_ivp(lambda s, y: p["f"](s, y, np), p["intervalo"], p["y0"],
                    method="DOP853", rtol=1e-13, atol=1e-13, dense_output=True)
    return ref.sol(t.ravel()).T

### 3.2 Runge–Kutta adaptativo (RK45)

O `solve_ivp` do SciPy anda passo a passo. Em cada passo, faz duas contas (de ordem 4 e de ordem 5); se elas discordam
mais do que a tolerância, refaz o passo menor. O `nfev` conta quantas vezes a regra `f` foi avaliada.

In [ ]:
# Resolve um problema com Runge-Kutta adaptativo (RK45 do SciPy, tolerância 1e-10).
# (Mesmo código do arquivo rk.py.)
import time

import numpy as np
from scipy.integrate import solve_ivp


def resolver(nome, tol=1e-10, repeticoes=1):
    """RK45 com tolerância `tol`. Roda `repeticoes` vezes e guarda o tempo mediano (o do meio)."""
    p = PROBLEMAS[nome]

    def f(t, y):
        # Mesma regra da ficha do problema, usando a caixa de ferramentas numpy
        return p["f"](t, y, np)

    tempos = []
    for _ in range(repeticoes):
        inicio = time.perf_counter()
        sol = solve_ivp(
            f,
            t_span=p["intervalo"],   # intervalo de tempo
            y0=p["y0"],              # ponto de partida
            method="RK45",
            rtol=tol,                # tolerância relativa
            atol=tol,                # tolerância absoluta
            dense_output=True,       # permite avaliar a solução em qualquer t
        )
        tempos.append(time.perf_counter() - inicio)
    tempo = float(np.median(tempos))

    # A régua comum: erro máximo numa grade de 1000 pontos, em todas as variáveis
    t = grade_da_regua(nome)
    y_rk = sol.sol(t.ravel()).T            # uma linha por tempo, uma coluna por variável
    erro = np.max(np.abs(y_rk - resposta_certa(nome, t)))

    return {"erro": erro, "tempo_s": tempo, "aval_f": sol.nfev, "passos": len(sol.t) - 1,
            "t_passos": sol.t, "y_passos": sol.y.T}

### 3.3 PINN mínima

A rede (2 camadas de 32 neurônios) chuta a curva inteira. Em 200 pontos de colocação, os "fiscais" medem quanto a
derivada da rede desobedece a regra; mais um fiscal confere o ponto de partida. O Adam gira os botões da rede para
diminuir a soma dessas multas. Cada iteração avalia `f` nos 200 pontos.

In [ ]:
# Resolve um problema com uma PINN mínima (PyTorch).
# (Mesmo código do arquivo pinn.py.)
import time

import numpy as np
import torch


def treinar(nome, n_iteracoes=5000, momentos_foto=(), peso_inicio=1.0, amarrar=False):
    """Treina uma PINN para o problema `nome`.

    A semente NÃO é fixada aqui: quem chama fixa uma única vez, no começo do programa
    (regra de reprodutibilidade do trabalho).
    peso_inicio : quanto vale a multa do ponto de partida (remédio 1: dar mais voz a esse fiscal).
    amarrar     : se True, a resposta é montada como y0 + s * rede(s), que vale y0 em s = 0 de qualquer jeito
                  (remédio 2: amarrar a curva no ponto de partida).
    """
    p = PROBLEMAS[nome]
    a, b = p["intervalo"]
    n_var = len(p["y0"])          # quantas variáveis (1 ou 2)
    y0 = torch.tensor([p["y0"]])

    # A rede neural: uma função y(t) cheia de botões (pesos) ajustáveis.
    # Entra 1 número (o tempo), passa por 2 camadas de 32 neurônios, sai 1 número por variável.
    corpo = torch.nn.Sequential(
        torch.nn.Linear(1, 32), torch.nn.Tanh(),
        torch.nn.Linear(32, 32), torch.nn.Tanh(),
        torch.nn.Linear(32, n_var),
    )

    def rede(t):
        s = (t - a) / (b - a)     # a rede enxerga o tempo "arrumado" entre 0 e 1 (em [0, 1] não muda nada)
        if amarrar:
            return y0 + s * corpo(s)
        return corpo(s)

    # Pontos de colocação: onde os "fiscais" conferem se a rede obedece a regra.
    # requires_grad=True permite calcular a derivada dy/dt automaticamente.
    t_col = torch.linspace(a, b, 200).reshape(-1, 1).requires_grad_(True)
    t0 = torch.full((1, 1), a)

    otimizador = torch.optim.Adam(corpo.parameters(), lr=1e-3)
    grade_foto = torch.linspace(a, b, 201).reshape(-1, 1)
    fotos = []

    inicio = time.perf_counter()
    for it in range(n_iteracoes):
        otimizador.zero_grad()

        y = rede(t_col)
        # Derivada de cada variável em relação a t (diferenciação automática)
        dy_dt = torch.cat([
            torch.autograd.grad(y[:, k:k + 1], t_col, torch.ones_like(t_col), create_graph=True)[0]
            for k in range(n_var)
        ], dim=1)

        multa_regra = ((dy_dt - p["f"](t_col, y, torch)) ** 2).mean()   # quanto desobedece a regra
        multa_inicio = ((rede(t0) - y0) ** 2).mean()                     # quanto erra o ponto de partida
        loss = multa_regra + peso_inicio * multa_inicio

        if it in momentos_foto:
            with torch.no_grad():
                fotos.append({"it": it, "regra": multa_regra.item(), "inicio": multa_inicio.item(),
                              "y": rede(grade_foto).tolist()})

        loss.backward()     # calcula para que lado girar cada botão
        otimizador.step()   # gira os botões um pouquinho
    tempo = time.perf_counter() - inicio

    # A régua comum: erro máximo numa grade de 1000 pontos, em todas as variáveis
    t = grade_da_regua(nome)
    with torch.no_grad():
        y_pinn = rede(torch.tensor(t, dtype=torch.float32)).numpy()
    erro = float(np.max(np.abs(y_pinn - resposta_certa(nome, t))))

    return {"erro": erro, "tempo_s": tempo, "aval_f": n_iteracoes * len(t_col),
            "loss_final": loss.item(), "fotos": fotos,
            "t_regua": t.ravel(), "y_regua": y_pinn}

### 3.4 A comparação

Roda os dois métodos nos três problemas, no mesmo computador e com a mesma régua. A semente aleatória é fixada uma única
vez, antes de todos os treinos, e os treinos rodam sempre na mesma ordem. Leva uns 3 minutos.

In [ ]:
# A comparação: os dois métodos, nos três problemas, no mesmo computador e com a mesma régua de erro.
# (Mesma sequência do arquivo exercicio1.py.)
SEMENTE = 0
torch.manual_seed(SEMENTE)   # semente fixada UMA única vez, antes de todos os treinos

TOLERANCIAS_CURVA = [10.0 ** -k for k in range(3, 13)]   # 1e-3, 1e-4, ..., 1e-12
REPETICOES_TEMPO_RK = 20                                # o RK é rápido: mede 20 vezes e usa a mediana

# Rótulos curtos de cada problema (usados nas tabelas e figuras)
ROTULO = {
    "rigido": "(i) y' = -50y",
    "oscilante": "(ii) y' = cos(20t)",
    "coelhos_raposas": "(iii) coelhos e raposas",
}

# Os treinos de PINN, NA ORDEM em que rodam (a ordem importa: todos usam a mesma sequência de sorteios).
# (identificador, problema, descrição do método, opções de treino)
TREINOS_PINN = [
    ("pinn_i", "rigido", "PINN, 5 mil iterações", {}),
    ("pinn_ii", "oscilante", "PINN, 5 mil iterações", {}),
    ("pinn_iii", "coelhos_raposas", "PINN, 5 mil iterações", {}),
    ("pinn_iii_20k", "coelhos_raposas", "PINN, 20 mil iterações", {"n_iteracoes": 20000}),
    ("pinn_i_peso", "rigido", "PINN, fiscal do início x 100", {"peso_inicio": 100.0}),
    ("pinn_i_amarrada", "rigido", "PINN, curva amarrada no início", {"amarrar": True}),
]

res = {"rk": {}, "rk_curva": {}, "pinn": {}}

# 1) Runge-Kutta com a tolerância do enunciado (1e-10), nos três problemas
for nome in PROBLEMAS:
    r = resolver(nome, tol=1e-10, repeticoes=REPETICOES_TEMPO_RK)
    res["rk"][nome] = r
    print(f"RK45 em {ROTULO[nome]}: erro {r['erro']:.2e}, {r['tempo_s'] * 1000:.1f} ms, {r['aval_f']} avaliações de f")

# 2) Curva custo x precisão do RK45 (várias tolerâncias)
for nome in PROBLEMAS:
    res["rk_curva"][nome] = [resolver(nome, tol=tol, repeticoes=5) for tol in TOLERANCIAS_CURVA]

# 3) PINNs (cada treino leva uns 20 segundos; o de 20 mil iterações, mais de 1 minuto)
for ident, nome, desc, opcoes in TREINOS_PINN:
    r = treinar(nome, **opcoes)
    r["problema"], r["descricao"] = nome, desc
    res["pinn"][ident] = r
    aval = f"{r['aval_f']:,}".replace(",", ".")   # ponto de milhar: 1.000.000
    print(f"{desc} em {ROTULO[nome]}: erro {r['erro']:.2e}, {r['tempo_s']:.1f} s, {aval} avaliações de f")

### 3.5 Tabelas e figuras

Grava tudo em `resultados/`, a partir dos números calculados acima.

In [ ]:
# Grava em resultados/ as tabelas (csv, md, tex), as figuras e os números do relatório.
# Aqui não se calcula nada de novo: só se desenha e se formata o que está em "res".
# Esse código de desenho (matplotlib) é longo e está no arquivo exercicio1.py; ele é reaproveitado aqui.
sys.path.insert(0, os.getcwd())   # para o Python achar o exercicio1.py desta pasta
import exercicio1

os.makedirs(exercicio1.PASTA, exist_ok=True)
exercicio1.salvar_tabelas(res)
exercicio1.figura_solucoes(res)
exercicio1.figura_custo_precisao(res)
exercicio1.figura_custo_compacta(res)
exercicio1.figura_remedios(res)
exercicio1.figura_fase(res)
exercicio1.salvar_numeros_tex(res)
exercicio1.salvar_json(res)
print("Arquivos em resultados/:", ", ".join(sorted(os.listdir(exercicio1.PASTA))))

## 4. Resultados

Erro máximo numa grade de 1000 pontos (mesma régua para os dois métodos), tempo e número de avaliações de f.

In [ ]:
from IPython.display import Image, Markdown, display

display(Markdown(open("resultados/tabela.md", encoding="utf-8").read()))

In [ ]:
FIGURAS = [
    ("fig2_custo_precisao.png", "Custo × precisão: curva do RK45 (várias tolerâncias) e pontos das PINNs. Abaixo e à esquerda é melhor."),
    ("fig1_solucoes.png", "Resposta certa, passos do RK45 e PINN com 5 mil iterações."),
    ("fig3_remedios_rigido.png", "Problema (i): a falha da PINN e dois remédios para a condição inicial."),
    ("fig4_fase_coelhos_raposas.png", "Problema (iii) no plano de fase: com 5 mil iterações a PINN espirala para o equilíbrio."),
]
for arquivo, legenda in FIGURAS:
    display(Markdown(f"**{legenda}**"))
    display(Image(filename=os.path.join("resultados", arquivo), width=900))

## 5. (Opcional) Um problema de cada vez

As funções `resolver` (RK45) e `treinar` (PINN) também rodam sozinhas. Os nomes dos problemas são `rigido` (y′ = −50y),
`oscilante` (y′ = cos(20t)) e `coelhos_raposas` (Lotka–Volterra).

In [ ]:
# Um problema de cada vez. Troque o nome por "rigido", "oscilante" ou "coelhos_raposas".
nome = "oscilante"

r = resolver(nome)
print(f"Runge-Kutta (RK45) em {PROBLEMAS[nome]['descricao']}")
print(f"  erro máximo      : {r['erro']:.2e}")
print(f"  tempo            : {r['tempo_s'] * 1000:.1f} ms")
print(f"  avaliações de f  : {r['aval_f']}")

torch.manual_seed(0)
r = treinar(nome)
print(f"PINN em {PROBLEMAS[nome]['descricao']}")
print(f"  erro máximo      : {r['erro']:.2e}")
print(f"  tempo            : {r['tempo_s']:.1f} s")
print(f"  avaliações de f  : {r['aval_f']}  (5000 iterações x 200 pontos)")